# 04. Evaluation — Confusion Matrix & Accuracy
# 04. 평가 — 혼동 행렬 & 정확도

**Goal / 목표**
- Evaluate the trained Grayspot model on the labeled dataset
- 학습된 Grayspot 모델을 라벨링 데이터셋으로 평가한다
- Compute Accuracy, F1, MAE, Confusion Matrix, Confidence Distribution
- 정확도, F1, MAE, 혼동 행렬, 신뢰도 분포를 계산한다
- Output interactive HTML reports to data_set/reports/
- data_set/reports/ 에 인터랙티브 HTML 리포트를 저장한다

**Compatibility / 호환성**
- Shares IMAGE_SIZE=128, cv2 loading, folder structure with 02/03/05 notebooks
- 02/03/05 노트북과 IMAGE_SIZE=128, cv2 로딩, 폴더 구조를 공유한다
- Loads baseline_{channel}.pt saved by 03_training.ipynb
- 03_training.ipynb 가 저장한 baseline_{channel}.pt 를 로드한다

**Folder Structure / 폴더 구조**
```
CMYK_MAIN/
  data_set/
    labeled/{channel}/{level}/*.png
    models/baseline_C.pt   <- 03_training.ipynb output
    models/baseline_Y.pt
    reports/               <- this notebook output
    labels_v0.csv
```

**Execution Order / 실행 순서**
1. Import libraries
2. Path & settings
3. Load labels
4. Dataset & DataLoader
5. Load model
6. Run inference
7. Compute metrics
8. Confusion matrix (HTML)
9. Evaluation dashboard (HTML)
10. Per-class F1 chart (HTML)
11. MAE heatmap (HTML)
12. Misclassified samples
13. Confidence distribution (HTML)
14. Export CSV + JSON
15. Phase 3 feedback decision
16. Output summary

---
## 1. Import Libraries / 라이브러리 임포트

In [3]:
# 필요 패키지 설치 (최초 1회) / Install required packages (first time only)
# !pip install torch torchvision scikit-learn plotly pandas numpy opencv-python Pillow

import csv
import json
import random
import sys
import warnings
import webbrowser
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple

warnings.filterwarnings("ignore")

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Python version guard / 파이썬 버전 확인
assert sys.version_info[:2] == (3, 11), f"Python 3.11.x required. Found: {sys.version}"

# Device setup — same logic as 02/03/05
# 디바이스 설정 — 02/03/05 와 동일한 방식
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)

# Reproducibility / 재현성 시드 고정
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Libraries loaded / 라이브러리 로드 완료")
print(f"PyTorch version / 버전: {torch.__version__}")
print(f"Device / 디바이스: {device}")

Libraries loaded / 라이브러리 로드 완료
PyTorch version / 버전: 2.11.0
Device / 디바이스: mps


---
## 2. Path & Settings / 경로 및 설정

> Edit only this cell — all paths propagate automatically.
> 이 셀만 수정하면 모든 경로가 자동으로 반영된다.

In [4]:
ROOT_DIR = Path("../..").resolve()  # CMYK_MAIN/
LABELED_DIR = ROOT_DIR / "data_set" / "labeled"  # labeled/{channel}/{level}/*.png
MODELS_DIR = ROOT_DIR / "data_set" / "models"  # baseline_C.pt etc.
LABELS_CSV = ROOT_DIR / "data_set" / "labels_v0.csv"
OUTPUT_DIR = ROOT_DIR / "data_set" / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Checkpoint to evaluate / 평가할 체크포인트
# Select one checkpoint saved by 03_training.ipynb
# 03_training.ipynb 가 저장한 체크포인트 중 하나를 선택한다
#   MODELS_DIR / 'baseline_C.pt'
#   MODELS_DIR / 'baseline_Y.pt'
#   MODELS_DIR / 'baseline_M.pt'
#   MODELS_DIR / 'baseline_K.pt'
CHECKPOINT = MODELS_DIR / "baseline_C.pt"

# Data settings — must match 03_training.ipynb
# 데이터 설정 — 03_training.ipynb 와 반드시 일치해야 한다
CHANNELS = ["Y", "M", "C", "K"]
NUM_LEVELS = 6  # Level 0 ~ 5
IMAGE_SIZE = 128  # same as 02/03/05 / 02/03/05 와 동일
BATCH_SIZE = 32
BACKBONE_NAME = "efficientnet_b0"  # must match 03_training.ipynb
COLOR_COLUMNS = {"Y": "Y", "M": "M", "C": "C", "K": "K"}

# Performance targets / 성능 목표 (PRD 1.4)
TARGET_OVERALL_ACC = 0.90
TARGET_PER_CLASS_F1 = 0.80
TARGET_PER_COLOR_ACC = 0.85
TARGET_MAE = 0.50

# Confidence thresholds / 신뢰도 임계값 (PRD 14.2)
CONF_THRESH_AUTO = 0.8
CONF_THRESH_WARN = 0.5
CONF_THRESH_MANUAL = 0.3

# Visualization colors / 시각화 색상
LEVEL_COLORS = ["#2ecc71", "#f1c40f", "#e67e22", "#e74c3c", "#9b59b6", "#1a1a2e"]
CMYK_COLORS = {"Y": "#f5e642", "M": "#e91e8c", "C": "#00b4d8", "K": "#444444"}

# Validate paths / 경로 유효성 확인
assert LABELED_DIR.exists(), f"labeled/ not found: {LABELED_DIR}"
assert CHECKPOINT.exists(), f"Checkpoint not found: {CHECKPOINT}"
assert LABELS_CSV.exists(), f"labels_v0.csv not found: {LABELS_CSV}"

print(f"ROOT_DIR    : {ROOT_DIR}")
print(f"LABELED_DIR : {LABELED_DIR}")
print(f"CHECKPOINT  : {CHECKPOINT}")
print(f"OUTPUT_DIR  : {OUTPUT_DIR}")
print()
print("Labeled folder counts / 라벨링 폴더 샘플 수:")
for ch in CHANNELS:
    for lv in range(NUM_LEVELS):
        folder = LABELED_DIR / ch / str(lv)
        count = len(list(folder.glob("*.png"))) if folder.exists() else 0
        print(f"  [{ch}] Level {lv}: {count}")

ROOT_DIR    : /Users/yangjin-hyeong/Desktop/CMYK_MAIN
LABELED_DIR : /Users/yangjin-hyeong/Desktop/CMYK_MAIN/data_set/labeled
CHECKPOINT  : /Users/yangjin-hyeong/Desktop/CMYK_MAIN/data_set/models/baseline_C.pt
OUTPUT_DIR  : /Users/yangjin-hyeong/Desktop/CMYK_MAIN/data_set/reports

Labeled folder counts / 라벨링 폴더 샘플 수:
  [Y] Level 0: 7
  [Y] Level 1: 23
  [Y] Level 2: 39
  [Y] Level 3: 38
  [Y] Level 4: 5
  [Y] Level 5: 0
  [M] Level 0: 14
  [M] Level 1: 137
  [M] Level 2: 228
  [M] Level 3: 472
  [M] Level 4: 200
  [M] Level 5: 73
  [C] Level 0: 0
  [C] Level 1: 16
  [C] Level 2: 34
  [C] Level 3: 89
  [C] Level 4: 32
  [C] Level 5: 2
  [K] Level 0: 5
  [K] Level 1: 16
  [K] Level 2: 71
  [K] Level 3: 231
  [K] Level 4: 121
  [K] Level 5: 2


---
## 3. Load Labels / 라벨 로드

Filename convention: scan_{num}_{color}_{num}.png
파일명 규칙: scan_{번호}_{색상}_{번호}.png

Each file exists only in the folder matching its color.
각 파일은 파일명에 포함된 색상 폴더에만 실제로 존재한다.
e.g. scan_001_C_0007.png -> labeled/C/{level}/ only

In [5]:
def extract_color_from_filename(fname: str) -> Optional[str]:
    """
    Extract color code from filename.
    파일명에서 색상 코드를 추출한다.
    e.g. 'scan_001_C_0007.png' -> 'C'
    """
    for part in Path(fname).stem.split("_"):
        if part in ("Y", "M", "C", "K"):
            return part
    return None


def load_labels(labels_csv: Path, color_columns: dict) -> pd.DataFrame:
    """
    Load label CSV and convert to long-format DataFrame.
    라벨 CSV를 로드하여 long-format DataFrame으로 변환한다.

    Original format (wide) / 원본 형식:
        filename, C, M, Y, K

    Return format (long) / 반환 형식:
        filename, color, level

    Only the label matching the color in the filename is kept.
    파일명의 색상과 일치하는 라벨만 유효하다.
    """
    df = pd.read_csv(labels_csv)
    print(f"CSV loaded / 로드: {len(df)} rows, columns: {list(df.columns)}")

    records = []
    skipped = 0
    for _, row in df.iterrows():
        color = extract_color_from_filename(str(row["filename"]))
        if color is None:
            skipped += 1
            continue
        col = color_columns.get(color)
        if col is None or col not in df.columns:
            skipped += 1
            continue
        records.append(
            {
                "filename": row["filename"],
                "color": color,
                "level": int(row[col]),
            }
        )

    long_df = pd.DataFrame(records)
    print(f"Long-format rows / 변환 결과: {len(long_df)}  (skipped / 제외: {skipped})")
    print(long_df.groupby(["color", "level"]).size().unstack(fill_value=0))
    return long_df


df_labels = load_labels(LABELS_CSV, COLOR_COLUMNS)
print(f"\nTotal samples / 총 샘플: {len(df_labels)}")
df_labels.head()

CSV loaded / 로드: 1855 rows, columns: ['filename', 'C', 'M', 'Y', 'K']
Long-format rows / 변환 결과: 1855  (skipped / 제외: 0)
level   0    1    2    3    4   5
color                            
C       0   16   34   89   32   2
K       5   16   71  231  121   2
M      14  137  228  472  200  73
Y       7   23   39   38    5   0

Total samples / 총 샘플: 1855


,filename,color,level
0,scan_001_C_0007.png,C,2
1,scan_002_C_0019.png,C,1
2,scan_002_C_0025.png,C,1
3,scan_002_C_0031.png,C,3
4,scan_002_K_0008.png,K,3


---
## 4. Dataset & DataLoader / 데이터셋 구성

Image loading follows 02/03/05 — cv2, no torchvision transforms.
이미지 로딩은 02/03/05 방식을 따른다 — cv2 사용, torchvision transforms 미사용.

In [6]:
class EvalDataset(Dataset):
    """
    Evaluation dataset. Loads images with cv2, same as 03_training.ipynb.
    평가 데이터셋. 03_training.ipynb 와 동일하게 cv2 로 이미지를 로드한다.

    Folder structure / 폴더 구조:
        labeled/{color}/{level}/{filename}

    Since load_labels() already filters by filename color,
    each file is looked up directly in its own color/level folder.
    load_labels() 에서 파일명 색상 기준으로 필터링했으므로
    각 파일은 자신의 색상/레벨 폴더에서 직접 찾는다.
    """

    def __init__(self, df: pd.DataFrame, patch_dir: Path, image_size: int):
        self.df = df.reset_index(drop=True)
        self.patch_dir = Path(patch_dir)
        self.image_size = image_size

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        color = row["color"]
        fname = row["filename"]
        level = int(row["level"])

        img_path = self.patch_dir / color / str(level) / fname

        if not img_path.exists():
            raise FileNotFoundError(f"Image not found / 이미지 없음: {img_path}")

        # cv2 loading — same as 02/03/05 / 02/03/05 와 동일한 cv2 로딩
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.image_size, self.image_size))
        img = img.astype(np.float32) / 255.0  # [0,1] normalize

        # (H, W, 3) -> (3, H, W) tensor
        tensor = torch.tensor(img).permute(2, 0, 1).float()

        return tensor, level, fname


print("EvalDataset defined / EvalDataset 정의 완료")

EvalDataset defined / EvalDataset 정의 완료


---
## 5. Load Model / 모델 로드

Model structure must match 03_training.ipynb exactly.
모델 구조는 03_training.ipynb 와 정확히 일치해야 한다.

In [7]:
class ClassifierHead(nn.Module):
    """
    Classifier head — identical to 03_training.ipynb.
    분류기 헤드 — 03_training.ipynb 와 동일.

    Backbone output -> Linear(256) -> BN -> ReLU -> Dropout -> Linear(6)
    """

    def __init__(
        self,
        in_dim: int,
        hidden_dim: int = 256,
        num_classes: int = 6,
        dropout: float = 0.3,
    ):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class GrayspotModel(nn.Module):
    """
    Full model — identical to 03_training.ipynb.
    전체 모델 — 03_training.ipynb 와 동일.
    """

    def __init__(self, backbone: nn.Module, feature_dim: int, num_classes: int = 6):
        super().__init__()
        self.backbone = backbone
        self.head = ClassifierHead(feature_dim, num_classes=num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.backbone(x)
        return self.head(features)


def build_model(
    backbone_name: str, num_classes: int, checkpoint: Path, device: torch.device
) -> GrayspotModel:
    """
    Build model and load checkpoint saved by 03_training.ipynb.
    03_training.ipynb 가 저장한 체크포인트를 로드하여 모델을 구성한다.
    """
    # Build backbone — same logic as 03_training.ipynb
    # 백본 구성 — 03_training.ipynb 와 동일한 방식
    if backbone_name == "efficientnet_b0":
        from torchvision.models import EfficientNet_B0_Weights

        backbone = models.efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        feature_dim = backbone.classifier[1].in_features  # 1280
        backbone.classifier = nn.Identity()

    elif backbone_name == "resnet50":
        from torchvision.models import ResNet50_Weights

        backbone = models.resnet50(weights=ResNet50_Weights.DEFAULT)
        feature_dim = backbone.fc.in_features  # 2048
        backbone.fc = nn.Identity()

    else:
        raise ValueError(f"Unsupported backbone: {backbone_name}")

    model = GrayspotModel(backbone, feature_dim, num_classes=num_classes)

    # Load checkpoint saved by 03_training.ipynb (model.state_dict())
    # 03_training.ipynb 가 model.state_dict() 로 저장한 파일을 로드
    state = torch.load(str(checkpoint), map_location="cpu")
    # Support both raw state_dict and wrapped dict
    # state_dict 직접 저장과 래핑된 dict 모두 지원
    if isinstance(state, dict) and "model_state_dict" in state:
        state = state["model_state_dict"]
    model.load_state_dict(state, strict=False)
    print(f"Checkpoint loaded / 체크포인트 로드: {checkpoint.name}")

    model = model.to(device)
    model.eval()
    return model


model = build_model(BACKBONE_NAME, NUM_LEVELS, CHECKPOINT, device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Backbone: {BACKBONE_NAME}")
print(f"Total parameters / 총 파라미터: {total_params:,}")
print(f"Device / 디바이스: {device}")

Checkpoint loaded / 체크포인트 로드: baseline_C.pt
Backbone: efficientnet_b0
Total parameters / 총 파라미터: 4,337,538
Device / 디바이스: mps


---
## 6. Run Inference / 추론 실행

In [8]:
def open_in_browser(path) -> None:
    """
    Open saved HTML file in default browser.
    저장된 HTML 파일을 기본 브라우저로 연다.
    Works on Windows and macOS via file:// URI.
    file:// URI 방식으로 Windows/macOS 모두 동작한다.
    """
    webbrowser.open(Path(path).resolve().as_uri())


@torch.no_grad()
def run_inference(
    model: GrayspotModel,
    loader: DataLoader,
    device: torch.device,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, List[str]]:
    """
    Run inference over the full DataLoader.
    DataLoader 전체에 대해 추론을 실행한다.

    Returns / 반환:
        y_true      : true labels (N,)
        y_pred      : predicted labels (N,)
        confidences : max softmax confidence (N,)
        filenames   : image filename list
    """
    model.eval()
    all_true, all_pred, all_conf, all_fnames = [], [], [], []

    for batch_imgs, batch_labels, batch_fnames in loader:
        batch_imgs = batch_imgs.to(device, non_blocking=True)
        logits = model(batch_imgs)
        probs = F.softmax(logits, dim=1)
        conf, preds = probs.max(dim=1)

        all_true.extend(batch_labels.numpy())
        all_pred.extend(preds.cpu().numpy())
        all_conf.extend(conf.cpu().numpy())
        all_fnames.extend(batch_fnames)

    return (
        np.array(all_true),
        np.array(all_pred),
        np.array(all_conf),
        all_fnames,
    )


# Run inference per channel / 채널별 추론 실행
results: Dict[str, dict] = {}

for color in CHANNELS:
    df_color = df_labels[df_labels["color"] == color].reset_index(drop=True)

    if len(df_color) == 0:
        print(f"[{color}] No samples — skipping / 샘플 없음 — 건너뜀")
        continue

    ds = EvalDataset(df_color, LABELED_DIR, IMAGE_SIZE)
    loader = DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,  # 0 = safe on Windows/macOS
        pin_memory=(device.type == "cuda"),
    )

    y_true, y_pred, confs, fnames = run_inference(model, loader, device)
    results[color] = {
        "y_true": y_true,
        "y_pred": y_pred,
        "confidences": confs,
        "filenames": fnames,
    }
    acc = accuracy_score(y_true, y_pred)
    print(f"[{color}] {len(y_true):4d} samples | Accuracy: {acc:.4f}")

active_channels = [c for c in CHANNELS if c in results]
print(f"\nInference complete / 추론 완료 — channels: {active_channels}")

[Y]  112 samples | Accuracy: 0.3393
[M] 1124 samples | Accuracy: 0.3923
[C]  173 samples | Accuracy: 0.2023
[K]  446 samples | Accuracy: 0.3049

Inference complete / 추론 완료 — channels: ['Y', 'M', 'C', 'K']


---
## 7. Compute Metrics / 지표 계산

In [9]:
def compute_metrics(
    y_true: np.ndarray, y_pred: np.ndarray, num_classes: int = NUM_LEVELS
) -> dict:
    """
    Compute classification and ordinal metrics.
    분류 및 순서형 지표를 계산한다.
    """
    labels = list(range(num_classes))
    accuracy = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, labels=labels, average=None, zero_division=0)
    rec = recall_score(y_true, y_pred, labels=labels, average=None, zero_division=0)
    f1 = f1_score(y_true, y_pred, labels=labels, average=None, zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    # MAE: treat Level as ordinal integer / Level을 순서형 정수로 취급 (PRD 1.4)
    mae = float(np.mean(np.abs(y_true.astype(float) - y_pred.astype(float))))

    per_class = [
        {"level": i, "precision": prec[i], "recall": rec[i], "f1": f1[i]}
        for i in labels
    ]
    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "mae": mae,
        "per_class": per_class,
    }


metrics: Dict[str, dict] = {}
for color in active_channels:
    metrics[color] = compute_metrics(results[color]["y_true"], results[color]["y_pred"])

all_true_combined = np.concatenate([results[c]["y_true"] for c in active_channels])
all_pred_combined = np.concatenate([results[c]["y_pred"] for c in active_channels])
metrics["overall"] = compute_metrics(all_true_combined, all_pred_combined)

# Print summary / 요약 출력
print("=== Performance Summary / 성능 요약 ===")
header = (
    f"{'Channel':>10}  {'Accuracy':>10}  {'Macro F1':>10}  {'MAE':>8}  Acc  F1  MAE"
)
print(header)
print("-" * len(header))

for ch in active_channels + ["overall"]:
    m = metrics[ch]
    tgt_acc = TARGET_OVERALL_ACC if ch == "overall" else TARGET_PER_COLOR_ACC
    acc_ok = "OK" if m["accuracy"] >= tgt_acc else "--"
    f1_ok = "OK" if m["macro_f1"] >= TARGET_PER_CLASS_F1 else "--"
    mae_ok = "OK" if m["mae"] <= TARGET_MAE else "--"
    print(
        f"{ch:>10}  {m['accuracy']:>10.4f}  {m['macro_f1']:>10.4f}  "
        f"{m['mae']:>8.4f}  {acc_ok:>3}  {f1_ok:>3}  {mae_ok:>3}"
    )

print()
print(f"Targets (PRD 1.4):")
print(f"  Overall Accuracy >= {TARGET_OVERALL_ACC:.0%}")
print(f"  Per-color Acc    >= {TARGET_PER_COLOR_ACC:.0%}")
print(f"  Per-class F1     >= {TARGET_PER_CLASS_F1:.2f}")
print(f"  MAE              <= {TARGET_MAE:.2f}")

print("\n=== Per-Class Performance (Overall) / 클래스별 성능 (전체) ===")
for pc in metrics["overall"]["per_class"]:
    ok = "OK" if pc["f1"] >= TARGET_PER_CLASS_F1 else "--"
    print(
        f"  Level {pc['level']}  Prec={pc['precision']:.4f}  Recall={pc['recall']:.4f}  F1={pc['f1']:.4f}  {ok}"
    )

=== Performance Summary / 성능 요약 ===
   Channel    Accuracy    Macro F1       MAE  Acc  F1  MAE
----------------------------------------------------------
         Y      0.3393      0.2305    0.7857   --   --   --
         M      0.3923      0.1909    0.8674   --   --   --
         C      0.2023      0.1307    1.2486   --   --   --
         K      0.3049      0.1408    0.9238   --   --   --
   overall      0.3504      0.2110    0.9116   --   --   --

Targets (PRD 1.4):
  Overall Accuracy >= 90%
  Per-color Acc    >= 85%
  Per-class F1     >= 0.80
  MAE              <= 0.50

=== Per-Class Performance (Overall) / 클래스별 성능 (전체) ===
  Level 0  Prec=0.0000  Recall=0.0000  F1=0.0000  --
  Level 1  Prec=0.2357  Recall=0.3438  F1=0.2797  --
  Level 2  Prec=0.2286  Recall=0.0215  F1=0.0393  --
  Level 3  Prec=0.4715  Recall=0.4988  F1=0.4848  --
  Level 4  Prec=0.2476  Recall=0.4274  F1=0.3135  --
  Level 5  Prec=0.2045  Recall=0.1169  F1=0.1488  --


---
## 8. Confusion Matrix / 혼동 행렬 (HTML)

In [10]:
def plot_confusion_matrix(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    title: str,
    normalize: bool = True,
    output_path: Optional[str] = None,
) -> go.Figure:
    """
    6x6 confusion matrix as a Plotly heatmap.
    6x6 혼동 행렬 Plotly 히트맵.
    """
    labels = list(range(NUM_LEVELS))
    level_names = [f"Level {i}" for i in labels]
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    if normalize:
        row_sums = cm.sum(axis=1, keepdims=True)
        z = np.where(row_sums > 0, cm / row_sums, 0.0)
        z_text = [[f"{v:.2f}" for v in row] for row in z]
        cbar_title = "Proportion"
        vmin, vmax = 0.0, 1.0
    else:
        z = cm.astype(float)
        z_text = [[str(v) for v in row] for row in cm]
        cbar_title = "Count"
        vmin, vmax = None, None

    # Reverse y-axis so Level 0 is at top / y축 반전으로 Level 0 상단 배치
    z_flip = z[::-1]
    z_text_flip = z_text[::-1]
    y_labels = level_names[::-1]

    fig = go.Figure(
        go.Heatmap(
            z=z_flip,
            x=level_names,
            y=y_labels,
            text=z_text_flip,
            texttemplate="%{text}",
            colorscale="Blues",
            zmin=vmin,
            zmax=vmax,
            colorbar=dict(title=cbar_title),
            hovertemplate="Predicted: %{x}<br>True: %{y}<br>Value: %{text}<extra></extra>",
        )
    )
    fig.update_layout(
        title=dict(text=title, font=dict(size=15)),
        xaxis_title="Predicted Level / 예측 레벨",
        yaxis_title="True Level / 실제 레벨",
        font=dict(family="Segoe UI", size=12),
        template="plotly_dark",
        width=600,
        height=520,
        margin=dict(l=40, r=40, t=60, b=40),
    )

    if output_path:
        fig.write_html(output_path, include_plotlyjs="cdn")
        print(f"Saved / 저장: {output_path}")

    return fig


for ch in active_channels + ["overall"]:
    yt = results[ch]["y_true"] if ch != "overall" else all_true_combined
    yp = results[ch]["y_pred"] if ch != "overall" else all_pred_combined
    acc = metrics[ch]["accuracy"]

    html_path = str(OUTPUT_DIR / f"cm_{ch}.html")
    plot_confusion_matrix(
        yt,
        yp,
        title=f"[{ch}] Confusion Matrix  Acc={acc:.4f}",
        normalize=True,
        output_path=html_path,
    )
    open_in_browser(html_path)

print("All confusion matrices saved / 혼동 행렬 저장 완료")

Saved / 저장: /Users/yangjin-hyeong/Desktop/CMYK_MAIN/data_set/reports/cm_Y.html
Saved / 저장: /Users/yangjin-hyeong/Desktop/CMYK_MAIN/data_set/reports/cm_M.html
Saved / 저장: /Users/yangjin-hyeong/Desktop/CMYK_MAIN/data_set/reports/cm_C.html
Saved / 저장: /Users/yangjin-hyeong/Desktop/CMYK_MAIN/data_set/reports/cm_K.html
Saved / 저장: /Users/yangjin-hyeong/Desktop/CMYK_MAIN/data_set/reports/cm_overall.html
All confusion matrices saved / 혼동 행렬 저장 완료


---
## 9. Evaluation Dashboard / 평가 대시보드 (HTML)

In [11]:
def plot_eval_dashboard(
    metrics: Dict[str, dict],
    channels: List[str],
    output_path: Optional[str] = None,
) -> go.Figure:
    """
    Gauge + Bar dashboard for Accuracy / Macro F1 / MAE.
    Accuracy / Macro F1 / MAE 를 Gauge + Bar 로 시각화한다.
    """
    m_all = metrics["overall"]
    valid_chs = [c for c in channels if c in metrics]

    fig = make_subplots(
        rows=2,
        cols=3,
        specs=[
            [{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}],
            [{"type": "bar", "colspan": 3}, None, None],
        ],
        subplot_titles=[
            "Overall Accuracy",
            "Overall Macro F1",
            "Overall MAE (lower is better / 낮을수록 좋음)",
            "Per-Color Accuracy / 색상별 정확도",
        ],
        vertical_spacing=0.18,
    )

    fig.add_trace(
        go.Indicator(
            mode="gauge+number",
            value=round(m_all["accuracy"] * 100, 2),
            number={"suffix": "%", "font": {"size": 30}},
            gauge={
                "axis": {"range": [0, 100]},
                "bar": {"color": "#50e3c2"},
                "threshold": {
                    "line": {"color": "#ff7aa2", "width": 3},
                    "value": TARGET_OVERALL_ACC * 100,
                },
                "bgcolor": "#0b1220",
            },
            title={"text": f"Target >= {TARGET_OVERALL_ACC:.0%}"},
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Indicator(
            mode="gauge+number",
            value=round(m_all["macro_f1"], 4),
            number={"font": {"size": 30}},
            gauge={
                "axis": {"range": [0, 1]},
                "bar": {"color": "#66d9ff"},
                "threshold": {
                    "line": {"color": "#ff7aa2", "width": 3},
                    "value": TARGET_PER_CLASS_F1,
                },
                "bgcolor": "#0b1220",
            },
            title={"text": f"Target >= {TARGET_PER_CLASS_F1:.2f}"},
        ),
        row=1,
        col=2,
    )

    fig.add_trace(
        go.Indicator(
            mode="gauge+number",
            value=round(m_all["mae"], 4),
            number={"font": {"size": 30}},
            gauge={
                "axis": {"range": [0, 3]},
                "bar": {"color": "#c792ea"},
                "threshold": {
                    "line": {"color": "#ffb347", "width": 3},
                    "value": TARGET_MAE,
                },
                "bgcolor": "#0b1220",
            },
            title={"text": f"Target <= {TARGET_MAE:.2f}"},
        ),
        row=1,
        col=3,
    )

    acc_vals = [metrics[c]["accuracy"] * 100 for c in valid_chs]
    fig.add_trace(
        go.Bar(
            x=valid_chs,
            y=acc_vals,
            marker_color=[CMYK_COLORS[c] for c in valid_chs],
            text=[f"{v:.2f}%" for v in acc_vals],
            textposition="outside",
            name="Accuracy",
        ),
        row=2,
        col=1,
    )

    fig.add_shape(
        type="line",
        x0=-0.5,
        x1=len(valid_chs) - 0.5,
        y0=TARGET_PER_COLOR_ACC * 100,
        y1=TARGET_PER_COLOR_ACC * 100,
        line=dict(color="#ff7aa2", dash="dash"),
        xref="x",
        yref="y",
        row=2,
        col=1,
    )
    fig.add_annotation(
        x=valid_chs[-1],
        y=TARGET_PER_COLOR_ACC * 100,
        text=f"Target {TARGET_PER_COLOR_ACC:.0%}",
        showarrow=False,
        yshift=10,
        row=2,
        col=1,
    )

    fig.update_layout(
        title=dict(
            text="Grayspot Evaluation Dashboard / 평가 대시보드", font=dict(size=17)
        ),
        template="plotly_dark",
        font=dict(family="Segoe UI", size=12),
        height=750,
        showlegend=False,
        margin=dict(l=40, r=40, t=80, b=40),
    )

    if output_path:
        fig.write_html(output_path, include_plotlyjs="cdn")
        print(f"Saved / 저장: {output_path}")

    return fig


dashboard_path = str(OUTPUT_DIR / "eval_dashboard.html")
plot_eval_dashboard(metrics, CHANNELS, output_path=dashboard_path)
open_in_browser(dashboard_path)

Saved / 저장: /Users/yangjin-hyeong/Desktop/CMYK_MAIN/data_set/reports/eval_dashboard.html


---
## 10. Per-Class F1 Chart / 클래스별 F1 차트 (HTML)

In [12]:
def plot_per_class_metrics(
    metrics: Dict[str, dict],
    output_path: Optional[str] = None,
) -> go.Figure:
    """
    Per-class Precision / Recall / F1 bar chart.
    클래스별 Precision / Recall / F1 막대 차트.
    """
    pc = metrics["overall"]["per_class"]
    levels = [f"Level {d['level']}" for d in pc]

    fig = go.Figure()
    fig.add_trace(
        go.Bar(name="F1", x=levels, y=[d["f1"] for d in pc], marker_color="#1976D2")
    )
    fig.add_trace(
        go.Bar(
            name="Precision",
            x=levels,
            y=[d["precision"] for d in pc],
            marker_color="#388E3C",
        )
    )
    fig.add_trace(
        go.Bar(
            name="Recall", x=levels, y=[d["recall"] for d in pc], marker_color="#F57C00"
        )
    )

    fig.add_hline(
        y=TARGET_PER_CLASS_F1,
        line_dash="dash",
        line_color="#ff7aa2",
        annotation_text=f"F1 Target >= {TARGET_PER_CLASS_F1:.2f}",
    )
    fig.update_layout(
        title=dict(
            text="Per-Class Metrics (Overall) / 클래스별 지표", font=dict(size=15)
        ),
        barmode="group",
        yaxis=dict(range=[0, 1.15], title="Score"),
        xaxis=dict(title="Level"),
        template="plotly_dark",
        font=dict(family="Segoe UI", size=12),
        legend=dict(orientation="h", y=1.1),
        height=480,
        margin=dict(l=40, r=40, t=80, b=40),
    )

    if output_path:
        fig.write_html(output_path, include_plotlyjs="cdn")
        print(f"Saved / 저장: {output_path}")

    return fig


per_class_path = str(OUTPUT_DIR / "per_class_metrics.html")
plot_per_class_metrics(metrics, output_path=per_class_path)
open_in_browser(per_class_path)

Saved / 저장: /Users/yangjin-hyeong/Desktop/CMYK_MAIN/data_set/reports/per_class_metrics.html


---
## 11. MAE Heatmap / MAE 히트맵 (HTML)

In [13]:
def plot_mae_heatmap(
    results: Dict[str, dict],
    channels: List[str],
    output_path: Optional[str] = None,
) -> go.Figure:
    """
    MAE heatmap by (color x true level).
    (색상 x 실제 레벨) 조합별 MAE 히트맵.
    """
    valid_chs = [c for c in channels if c in results]
    level_names = [f"Level {i}" for i in range(NUM_LEVELS)]
    mae_matrix = np.full((len(valid_chs), NUM_LEVELS), np.nan)
    count_matrix = np.zeros((len(valid_chs), NUM_LEVELS), dtype=int)

    for ci, color in enumerate(valid_chs):
        yt = results[color]["y_true"]
        yp = results[color]["y_pred"]
        for lv in range(NUM_LEVELS):
            mask = yt == lv
            if mask.sum() > 0:
                mae_matrix[ci, lv] = float(np.mean(np.abs(yt[mask] - yp[mask])))
                count_matrix[ci, lv] = int(mask.sum())

    annot = [
        [
            (
                f"{mae_matrix[r, c]:.2f}\n(n={count_matrix[r, c]})"
                if not np.isnan(mae_matrix[r, c])
                else "N/A"
            )
            for c in range(NUM_LEVELS)
        ]
        for r in range(len(valid_chs))
    ]

    fig = go.Figure(
        go.Heatmap(
            z=mae_matrix,
            x=level_names,
            y=valid_chs,
            text=annot,
            texttemplate="%{text}",
            colorscale="YlOrRd",
            zmin=0,
            zmax=2.0,
            colorbar=dict(title="MAE"),
            hovertemplate="Color: %{y}<br>Level: %{x}<br>MAE: %{z:.3f}<extra></extra>",
        )
    )
    fig.update_layout(
        title=dict(
            text=f"MAE per (Color x True Level) — Target <= {TARGET_MAE}",
            font=dict(size=15),
        ),
        xaxis=dict(title="True Level / 실제 레벨"),
        yaxis=dict(title="Color Channel / 색상 채널"),
        template="plotly_dark",
        font=dict(family="Segoe UI", size=12),
        height=360,
        margin=dict(l=40, r=40, t=60, b=40),
    )

    if output_path:
        fig.write_html(output_path, include_plotlyjs="cdn")
        print(f"Saved / 저장: {output_path}")

    return fig


mae_path = str(OUTPUT_DIR / "mae_heatmap.html")
plot_mae_heatmap(results, CHANNELS, output_path=mae_path)
open_in_browser(mae_path)

Saved / 저장: /Users/yangjin-hyeong/Desktop/CMYK_MAIN/data_set/reports/mae_heatmap.html


---
## 12. Misclassified Samples / 오분류 샘플

In [14]:
records: List[dict] = []

for color in active_channels:
    yt = results[color]["y_true"]
    yp = results[color]["y_pred"]
    confs = results[color]["confidences"]
    fnames = results[color]["filenames"]

    for i in np.where(yt != yp)[0]:
        records.append(
            {
                "filename": fnames[i],
                "color": color,
                "true_level": int(yt[i]),
                "pred_level": int(yp[i]),
                "confidence": round(float(confs[i]), 4),
                "correct": False,
                "error_gap": int(abs(yt[i] - yp[i])),
            }
        )

df_miss = pd.DataFrame(records)
if len(df_miss) > 0:
    df_miss = df_miss.sort_values(["error_gap", "confidence"], ascending=[False, True])
    print(f"Misclassified / 오분류: {len(df_miss)}")
    print(f'Max error gap / 최대 오류 간격: {df_miss["error_gap"].max()}')
    print("\nPer-color mismatch / 색상별 오분류:")
    print(df_miss.groupby("color").size().to_string())
else:
    print("No misclassifications / 오분류 없음")

# Save CSV — UTF-8 BOM for Windows Excel Korean support
# UTF-8 BOM: Windows Excel 한글 깨짐 방지
miss_path = OUTPUT_DIR / "misclassified_samples.csv"
df_miss.to_csv(miss_path, index=False, encoding="utf-8-sig")
print(f"\nSaved / 저장: {miss_path}")
df_miss.head(10)

Misclassified / 오분류: 1205
Max error gap / 최대 오류 간격: 4

Per-color mismatch / 색상별 오분류:
color
C    138
K    310
M    683
Y     74

Saved / 저장: /Users/yangjin-hyeong/Desktop/CMYK_MAIN/data_set/reports/misclassified_samples.csv


,filename,color,true_level,pred_level,confidence,correct,error_gap
92,scan_009_M_0005.png,M,0,4,0.2839,False,4
140,scan_024_M_0009.png,M,0,4,0.2881,False,4
95,scan_009_M_0008.png,M,0,4,0.3013,False,4
141,scan_024_M_0011.png,M,0,4,0.3017,False,4
786,scan_057_C_0019.png,C,1,5,0.3207,False,4
886,scan_140_C_0003.png,C,1,5,0.3280,False,4
764,scan_018_C_0007.png,C,1,5,0.3445,False,4
901,scan_002_K_0020.png,K,0,4,0.5023,False,4
903,scan_002_K_0024.png,K,0,4,0.5994,False,4
899,scan_002_K_0016.png,K,0,4,0.7647,False,4


In [15]:
def plot_mismatch_scatter(
    df_miss: pd.DataFrame,
    output_path: Optional[str] = None,
) -> go.Figure:
    """
    Misclassified samples scatter (color=channel, size=error_gap).
    오분류 샘플 scatter (color=채널, size=error_gap).
    """
    if len(df_miss) == 0:
        print(
            "No misclassified samples — skipping scatter / 오분류 없음 — scatter 건너뜀"
        )
        return go.Figure()

    fig = px.scatter(
        df_miss,
        x="true_level",
        y="pred_level",
        color="color",
        color_discrete_map=CMYK_COLORS,
        size="error_gap",
        size_max=20,
        hover_data=[
            "filename",
            "color",
            "true_level",
            "pred_level",
            "confidence",
            "error_gap",
        ],
        title="Misclassified Samples / 오분류 샘플 — True vs Predicted Level",
        labels={
            "true_level": "True Level / 실제 레벨",
            "pred_level": "Predicted Level / 예측 레벨",
            "color": "CMYK Channel",
        },
        template="plotly_dark",
        width=680,
        height=540,
    )
    # Diagonal: correct prediction line / 대각선: 정답 경계선
    fig.add_trace(
        go.Scatter(
            x=[0, NUM_LEVELS - 1],
            y=[0, NUM_LEVELS - 1],
            mode="lines",
            line=dict(color="gray", dash="dash", width=1),
            name="Correct boundary / 정답 경계",
            showlegend=True,
        )
    )
    fig.update_layout(
        font=dict(family="Segoe UI", size=12),
        margin=dict(l=40, r=40, t=60, b=40),
    )

    if output_path:
        fig.write_html(output_path, include_plotlyjs="cdn")
        print(f"Saved / 저장: {output_path}")

    return fig


scatter_path = str(OUTPUT_DIR / "misclassified_scatter.html")
plot_mismatch_scatter(df_miss, output_path=scatter_path)
open_in_browser(scatter_path)

Saved / 저장: /Users/yangjin-hyeong/Desktop/CMYK_MAIN/data_set/reports/misclassified_scatter.html


---
## 13. Confidence Distribution / 신뢰도 분포 (HTML, PRD 14.2)

In [16]:
def plot_confidence_distribution(
    results: Dict[str, dict],
    channels: List[str],
    output_path: Optional[str] = None,
) -> go.Figure:
    """
    Per-color confidence histogram (correct vs wrong).
    PRD 14.2 threshold vertical lines included.
    색상별 신뢰도 히스토그램. PRD 14.2 임계값 수직선 포함.
    """
    valid_chs = [c for c in channels if c in results]
    n_rows = (len(valid_chs) + 1) // 2

    fig = make_subplots(
        rows=n_rows,
        cols=2,
        subplot_titles=[f"[{c}]" for c in valid_chs],
        horizontal_spacing=0.10,
        vertical_spacing=0.18,
    )

    bins = dict(start=0, end=1, size=0.04)

    for i, color in enumerate(valid_chs):
        r = i // 2 + 1
        c = i % 2 + 1
        yt = results[color]["y_true"]
        yp = results[color]["y_pred"]
        cf = results[color]["confidences"]

        fig.add_trace(
            go.Histogram(
                x=cf[yt == yp],
                xbins=bins,
                name="Correct / 정답",
                marker_color="#4fc3f7",
                opacity=0.70,
                showlegend=(i == 0),
                legendgroup="correct",
            ),
            row=r,
            col=c,
        )

        fig.add_trace(
            go.Histogram(
                x=cf[yt != yp],
                xbins=bins,
                name="Wrong / 오답",
                marker_color="#ef5350",
                opacity=0.70,
                showlegend=(i == 0),
                legendgroup="wrong",
            ),
            row=r,
            col=c,
        )

        for thresh, col_color in [
            (CONF_THRESH_AUTO, "green"),
            (CONF_THRESH_WARN, "orange"),
            (CONF_THRESH_MANUAL, "red"),
        ]:
            fig.add_vline(
                x=thresh,
                line_dash="dash",
                line_color=col_color,
                line_width=1.5,
                row=r,
                col=c,
            )

    fig.update_layout(
        title=dict(
            text="Confidence Distribution / 신뢰도 분포 — Correct vs Wrong",
            font=dict(size=15),
        ),
        barmode="overlay",
        template="plotly_dark",
        font=dict(family="Segoe UI", size=12),
        height=640,
        margin=dict(l=40, r=40, t=80, b=40),
    )

    if output_path:
        fig.write_html(output_path, include_plotlyjs="cdn")
        print(f"Saved / 저장: {output_path}")

    return fig


conf_path = str(OUTPUT_DIR / "confidence_distribution.html")
plot_confidence_distribution(results, CHANNELS, output_path=conf_path)
open_in_browser(conf_path)

# Coverage table / 커버리지 테이블
all_confs = np.concatenate([results[c]["confidences"] for c in active_channels])
total = len(all_confs)

print("\n=== Confidence-Driven Coverage (PRD 14.2) ===")
print(f"{'Policy':32s} {'Threshold':>12s} {'Samples':>10s} {'Coverage':>10s}")
print("-" * 68)
for name, thresh in [
    ("Auto judgment / 자동 판정", CONF_THRESH_AUTO),
    ("Warn + auto / 경고 포함 자동", CONF_THRESH_WARN),
    ("Manual queue / 수동 검수 대기", CONF_THRESH_MANUAL),
]:
    n = int((all_confs >= thresh).sum())
    pct = n / total * 100
    print(f"{name:32s} {'>= ' + str(thresh):>12s} {n:>10d} {pct:>9.1f}%")
print(f"{'Total / 전체':32s} {'--':>12s} {total:>10d} {'100.0':>9s}%")

Saved / 저장: /Users/yangjin-hyeong/Desktop/CMYK_MAIN/data_set/reports/confidence_distribution.html

=== Confidence-Driven Coverage (PRD 14.2) ===
Policy                              Threshold    Samples   Coverage
--------------------------------------------------------------------
Auto judgment / 자동 판정                  >= 0.8        311      16.8%
Warn + auto / 경고 포함 자동                 >= 0.5       1056      56.9%
Manual queue / 수동 검수 대기                >= 0.3       1723      92.9%
Total / 전체                                 --       1855     100.0%


---
## 14. Export Results / 결과 내보내기 (CSV + JSON)

In [17]:
# evaluation_results.csv (PRD 8.2.2)
eval_rows = []
for color in active_channels:
    yt = results[color]["y_true"]
    yp = results[color]["y_pred"]
    confs = results[color]["confidences"]
    fnames = results[color]["filenames"]
    for i in range(len(yt)):
        eval_rows.append(
            {
                "filename": fnames[i],
                "color": color,
                "true_level": int(yt[i]),
                "pred_level": int(yp[i]),
                "confidence": round(float(confs[i]), 4),
                "correct": bool(yt[i] == yp[i]),
                "error_gap": int(abs(yt[i] - yp[i])),
            }
        )

df_eval = pd.DataFrame(eval_rows)
eval_csv = OUTPUT_DIR / "evaluation_results.csv"
df_eval.to_csv(eval_csv, index=False, encoding="utf-8-sig")
print(f"Saved / 저장: {eval_csv}  ({len(df_eval)} rows)")

# metrics_summary.json
metrics_export = {
    "meta": {
        "n_samples": int(len(df_labels)),
        "backbone": BACKBONE_NAME,
        "checkpoint": str(CHECKPOINT),
        "image_size": IMAGE_SIZE,
    },
    "targets": {
        "overall_accuracy": TARGET_OVERALL_ACC,
        "per_color_accuracy": TARGET_PER_COLOR_ACC,
        "per_class_f1": TARGET_PER_CLASS_F1,
        "mae": TARGET_MAE,
    },
    "global": {
        "accuracy": round(metrics["overall"]["accuracy"], 4),
        "macro_f1": round(metrics["overall"]["macro_f1"], 4),
        "mae": round(metrics["overall"]["mae"], 4),
        "acc_pass": bool(metrics["overall"]["accuracy"] >= TARGET_OVERALL_ACC),
        "f1_pass": bool(metrics["overall"]["macro_f1"] >= TARGET_PER_CLASS_F1),
        "mae_pass": bool(metrics["overall"]["mae"] <= TARGET_MAE),
    },
    "by_color": {
        color: {
            "accuracy": round(metrics[color]["accuracy"], 4),
            "macro_f1": round(metrics[color]["macro_f1"], 4),
            "mae": round(metrics[color]["mae"], 4),
            "acc_pass": bool(metrics[color]["accuracy"] >= TARGET_PER_COLOR_ACC),
        }
        for color in active_channels
    },
    "per_class_overall": [
        {
            "level": pc["level"],
            "precision": round(pc["precision"], 4),
            "recall": round(pc["recall"], 4),
            "f1": round(pc["f1"], 4),
            "f1_pass": bool(pc["f1"] >= TARGET_PER_CLASS_F1),
        }
        for pc in metrics["overall"]["per_class"]
    ],
}

json_path = OUTPUT_DIR / "metrics_summary.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(metrics_export, f, ensure_ascii=False, indent=2)
print(f"Saved / 저장: {json_path}")

Saved / 저장: /Users/yangjin-hyeong/Desktop/CMYK_MAIN/data_set/reports/evaluation_results.csv  (1855 rows)
Saved / 저장: /Users/yangjin-hyeong/Desktop/CMYK_MAIN/data_set/reports/metrics_summary.json


---
## 15. Phase 3 Feedback Decision / Phase 3 피드백 복귀 판단

PRD 3.3.2 feedback-loop decision logic
PRD 3.3.2 피드백 루프 판단 기준

In [18]:
print("=" * 65)
print("  Phase 3 Feedback Decision / Phase 3 피드백 복귀 판단")
print("=" * 65)

decisions: List[str] = []

# Check 1: per-color accuracy < 0.80 -> Phase 0
# 검사 1: 색상별 정확도 미달 -> Phase 0 복귀
for color in active_channels:
    acc = metrics[color]["accuracy"]
    if acc < 0.80:
        decisions.append(
            f"  [{color}] Accuracy {acc:.3f} < 0.80"
            " -> Phase 0 (retrain representation / 표현 재학습)"
        )

# Check 2: per-class F1 < 0.70 -> Phase 1
# 검사 2: 클래스별 F1 미달 -> Phase 1 복귀
for pc in metrics["overall"]["per_class"]:
    if pc["f1"] < 0.70:
        decisions.append(
            f"  Level {pc['level']} F1={pc['f1']:.3f} < 0.70"
            " -> Phase 1 (review level boundary / 레벨 경계 재검토)"
        )

# Check 3: overall MAE > 0.80 -> Phase 0
# 검사 3: MAE 초과 -> Phase 0 복귀
overall_mae = metrics["overall"]["mae"]
if overall_mae > 0.80:
    decisions.append(
        f"  Overall MAE {overall_mae:.3f} > 0.80"
        " -> Phase 0 (representation learning retry / 표현 학습 재시도)"
    )

overall_acc = metrics["overall"]["accuracy"]
overall_mf1 = metrics["overall"]["macro_f1"]
all_color_ok = all(
    metrics[c]["accuracy"] >= TARGET_PER_COLOR_ACC for c in active_channels
)
all_class_ok = all(
    pc["f1"] >= TARGET_PER_CLASS_F1 for pc in metrics["overall"]["per_class"]
)
mae_ok = overall_mae <= TARGET_MAE

if overall_acc >= TARGET_OVERALL_ACC and all_color_ok and all_class_ok and mae_ok:
    print("\n  All targets met -- TERMINATE Swing / 모든 목표 달성 -- Swing 종료")
    print(f"  Overall Accuracy : {overall_acc:.4f} >= {TARGET_OVERALL_ACC}")
    print(f"  Macro F1         : {overall_mf1:.4f} >= {TARGET_PER_CLASS_F1}")
    print(f"  MAE              : {overall_mae:.4f} <= {TARGET_MAE}")
elif not decisions:
    print("\n  No critical failures -- continue training or next cycle")
    print("  심각한 실패 없음 -- 학습 계속 또는 다음 사이클 진행")
else:
    print("\n  Action required / 조치 필요:")
    for d in decisions:
        print(d)

print()
print(f"  Overall Accuracy : {overall_acc:.4f}  (target >= {TARGET_OVERALL_ACC})")
print(f"  Overall Macro F1 : {overall_mf1:.4f}  (target >= {TARGET_PER_CLASS_F1})")
print(f"  Overall MAE      : {overall_mae:.4f}  (target <= {TARGET_MAE})")
print("=" * 65)

  Phase 3 Feedback Decision / Phase 3 피드백 복귀 판단

  Action required / 조치 필요:
  [Y] Accuracy 0.339 < 0.80 -> Phase 0 (retrain representation / 표현 재학습)
  [M] Accuracy 0.392 < 0.80 -> Phase 0 (retrain representation / 표현 재학습)
  [C] Accuracy 0.202 < 0.80 -> Phase 0 (retrain representation / 표현 재학습)
  [K] Accuracy 0.305 < 0.80 -> Phase 0 (retrain representation / 표현 재학습)
  Level 0 F1=0.000 < 0.70 -> Phase 1 (review level boundary / 레벨 경계 재검토)
  Level 1 F1=0.280 < 0.70 -> Phase 1 (review level boundary / 레벨 경계 재검토)
  Level 2 F1=0.039 < 0.70 -> Phase 1 (review level boundary / 레벨 경계 재검토)
  Level 3 F1=0.485 < 0.70 -> Phase 1 (review level boundary / 레벨 경계 재검토)
  Level 4 F1=0.314 < 0.70 -> Phase 1 (review level boundary / 레벨 경계 재검토)
  Level 5 F1=0.149 < 0.70 -> Phase 1 (review level boundary / 레벨 경계 재검토)
  Overall MAE 0.912 > 0.80 -> Phase 0 (representation learning retry / 표현 학습 재시도)

  Overall Accuracy : 0.3504  (target >= 0.9)
  Overall Macro F1 : 0.2110  (target >= 0.8)
  Overall MAE      : 

---
## 16. Output Summary / 산출물 목록

In [19]:
print("=" * 65)
print("Generated Outputs / 생성된 산출물 목록")
print("=" * 65)

output_files = [
    ("cm_Y.html", "Confusion Matrix [Y]"),
    ("cm_M.html", "Confusion Matrix [M]"),
    ("cm_C.html", "Confusion Matrix [C]"),
    ("cm_K.html", "Confusion Matrix [K]"),
    ("cm_overall.html", "Confusion Matrix [Overall]"),
    ("eval_dashboard.html", "Accuracy / F1 / MAE dashboard"),
    ("per_class_metrics.html", "Per-class Precision / Recall / F1"),
    ("mae_heatmap.html", "MAE heatmap (color x level)"),
    ("misclassified_scatter.html", "Misclassified samples scatter"),
    ("confidence_distribution.html", "Confidence distribution"),
    ("evaluation_results.csv", "Per-sample predictions / 샘플별 예측"),
    ("misclassified_samples.csv", "Misclassified sample list / 오분류 목록"),
    ("metrics_summary.json", "Aggregated metrics / 성능 요약"),
]

for fname, desc in output_files:
    status = "[OK]" if (OUTPUT_DIR / fname).exists() else "[  ]"
    print(f"  {status}  {fname:<45} {desc}")

print()
print("Next steps / 다음 단계 (PRD 3.3.2):")
print("  1. Open eval_dashboard.html and check target pass/fail")
print("     eval_dashboard.html 열고 목표 달성 여부 확인")
print("  2. Review misclassified_samples.csv sorted by error_gap desc")
print("     misclassified_samples.csv 를 error_gap 내림차순으로 검토")
print("  3. Cross-check with 06_embedding_viz.ipynb mismatch_samples.csv")
print("     06_embedding_viz.ipynb 의 mismatch_samples.csv 와 교차 확인")
print("  4. Phase 0 -> retrain backbone / Phase 1 -> refine labels")
print("     Phase 0 -> backbone 재학습 / Phase 1 -> 라벨 재정제")
print("  5. All targets met -> freeze model and proceed to Stage 5 Delivery")
print("     모든 목표 달성 -> 모델 고정 후 Stage 5 Delivery 진행")

Generated Outputs / 생성된 산출물 목록
  [OK]  cm_Y.html                                     Confusion Matrix [Y]
  [OK]  cm_M.html                                     Confusion Matrix [M]
  [OK]  cm_C.html                                     Confusion Matrix [C]
  [OK]  cm_K.html                                     Confusion Matrix [K]
  [OK]  cm_overall.html                               Confusion Matrix [Overall]
  [OK]  eval_dashboard.html                           Accuracy / F1 / MAE dashboard
  [OK]  per_class_metrics.html                        Per-class Precision / Recall / F1
  [OK]  mae_heatmap.html                              MAE heatmap (color x level)
  [OK]  misclassified_scatter.html                    Misclassified samples scatter
  [OK]  confidence_distribution.html                  Confidence distribution
  [OK]  evaluation_results.csv                        Per-sample predictions / 샘플별 예측
  [OK]  misclassified_samples.csv                     Misclassified sample list / 오분류 